In [82]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import re

In [70]:
with open('output/inv-trig/all-questions.md', 'r') as qf:
    questions = qf.readlines()
len(questions)

30

In [58]:
with open('output/inv-trig/rep-questions-approaches.md', 'r') as af:
    approaches = af.readlines()

In [59]:
len(approaches)

15

In [7]:
model = SentenceTransformer("BAAI/bge-m3")

In [71]:
embeddings = model.encode(questions)

In [72]:
# Cluster using KMeans
n_clusters = 15
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

In [73]:
# Find central question in each cluster
representatives = []
for cluster_id in range(n_clusters):
    idxs = [i for i, label in enumerate(labels) if label == cluster_id]
    cluster_embeddings = [embeddings[i] for i in idxs]
    sims = cosine_similarity([kmeans.cluster_centers_[cluster_id]], cluster_embeddings)
    central_idx = idxs[np.argmax(sims)]
    representatives.append((cluster_id, questions[central_idx]))

In [74]:
                                                                                                                                                                                    pattern = r"^([^.]+)"
print("Representative Questions from Each Cluster:\n")
cluster_map={}
for cluster_id, question in representatives:
    fixed_question = question.encode('unicode_escape').decode()
    match = re.search(pattern, fixed_question)
    if match:
        # Access the captured group using .group(1)
        extracted_substring = int(match.group(1).strip())
        cluster_map[extracted_substring] = fixed_question
    else:
        print("No match found.")
for id in sorted(cluster_map.keys()):
    print(cluster_map[id])

Representative Questions from Each Cluster:

1.    The value of \\( \\cos^{-1}\\left(\\frac{1}{\\sqrt{2}}\\right) - \\cos^{-1}\\left(-\\frac{\\sqrt{3}}{2}\\right) \\) is   (1) \\( 15^\\circ \\)   (2) \\( 75^\\circ \\)   (3) \\( 195^\\circ \\)   (4) \\( \\frac{3\\pi}{4} \\)  \n
5.    Which statement is True?   (1) \\( \\sin^{-1}(x) = x \\) if \\( x \\in R \\)   (2) \\( \\cos(\\cos^{-1}x) = x \\) if \\( x \\in [-1, 1] \\)   (3) \\( \\sec(\\sec^{-1}x) = x \\) if \\( x \\in [1, \\infty) \\)   (4) None of the above  \n
6.    If 3pi/2 < x < 5pi/2, then sin^{-1}(sin x) is equal to (1) x - 2pi (2) pi - x (3) 3pi - x (4) 2pi - x\n
8.    If \\( \\cot \\frac{n\\pi}{6} = n \\in \\mathbb{N} \\), then the maximum value of \\( n \\) is   (1) \\( 1 \\)   (2) \\( 5 \\)   (3) \\( 9 \\)   (4) \\( 6 \\)  \n
9.    If \\( 0 < x < \\frac{\\pi}{2} \\), then \\( \\left\\{ x \\cos(\\cot^{-1} x) + \\sin(\\cot^{-1} x) \\right\\}^{1/2} \\) is equal to:   (1) \\( \\frac{1}{\\sqrt{1 + x^2}} \\)   (2) \\( x \\)   (3)

In [81]:
with open('output/inv-trig/rep-questions.md', 'w') as rqf:
    for id in sorted(cluster_map.keys()):
        rqf.write(cluster_map[id] + '\n')

In [76]:
df_clusters = pd.DataFrame({
    "Question": questions,
    "Cluster": labels
})

In [77]:
pd.set_option("display.max_colwidth", 60)
print(df_clusters.sort_values("Cluster"))

                                                       Question  Cluster
1   2.    If \( 3 \tan x + \cot^{-1} x = 2 \), then \( x \) ...        0
20     21. Evaluate:   \[ \tan^{-1} x + \tan^{-1} 1, \quad \...        0
18     19. If \( \tan^{-1} x + \tan^{-1} y + \tan^{-1} z = 0...        0
10  11.    If \( \tan^{-1}\left(\frac{x + 1}{x}\right) = 4^\...        0
11     12.  Evaluate:   \[ \tan^{-1} x + \tan^{-1} 1 + \tan^...        0
13     14. Evaluate:   \[ \cot^{-1} 1 + \cot^{-1} \frac{9}{8...        0
22     23. If \( 2\tan(\cos^{-1} \theta)(\theta \neq 0) = 0 ...        1
21     22. Let \( \tan^{-1} \left( \frac{5\pi}{4} \right) = ...        1
9   10.    If \( x \neq 0 \), the value of \( \tan\left(\fra...        1
16     17. Find the value of:   \[ \tan^2 \left( \frac{\pi}{...        1
4   5.    Which statement is True?   (1) \( \sin^{-1}(x) = x...        2
0   1.    The value of \( \cos^{-1}\left(\frac{1}{\sqrt{2}}\...        3
7   8.    If \( \cot \frac{n\pi}{6} = n \in \mathbb

In [78]:
# Specify the cluster you want to print questions from
target_cluster = 9  # Replace with your desired cluster number

# Filter questions belonging to the target cluster
cluster_questions = df_clusters[df_clusters['Cluster'] == target_cluster]

print(f"Questions in Cluster {target_cluster}:\n")
for index, row in cluster_questions.iterrows():
    fixed_question = row['Question'].encode('unicode_escape').decode()
    print(fixed_question)

Questions in Cluster 9:

30.    The value of \\( \\sin^{-1}(\\sin 12) + \\cos^{-1}(\\cos 12) \\) is equal to  \n


In [65]:
approach_embeddings = model.encode(approaches)

In [66]:
with open('output/inv-trig/test/all-test-questions-approaches.md', 'r') as qf:
    test_approaches = qf.readlines()

In [67]:
# Encode test questions
test_approach_embeddings = model.encode(test_approaches)

closest_approaches_info = []

for i, test_approach_embedding in enumerate(test_approach_embeddings):
    similarities = cosine_similarity([test_approach_embedding], approach_embeddings)
    closest_idx = np.argmax(similarities)
    closest_approach = approaches[closest_idx]

    # Calculate similarity to closest question
    sim_to_closest_question = cosine_similarity(
        [test_approach_embedding], 
        [approach_embeddings[approaches.index(closest_approach)]]
    )[0][0]

    closest_approaches_info.append({
        "Test Approach": test_approaches[i],
        "Closest Approach": closest_approach,
        "Similarity to Closest Approach": sim_to_closest_question
    })

In [69]:
# Display the evaluation results
print("Test Approach | Closest Approach | Similarity")
for info in closest_approaches_info:
    print(f"{info['Test Approach'].strip()} | {info['Closest Approach']} | {info['Similarity to Closest Approach']}")

Test Approach | Closest Approach | Similarity
1: Set x = cos α and use cos 3α = 4cos^3α − 3cosα, determine α’s interval from x∈(−1,−1/2), reduce 3α by 2π to the principal arccos range [0,π], use arcsin x = π/2 − α, then add and simplify. | 30: Reduce 12 modulo 2π, identify which subintervals of the principal ranges arcsin∈[-π/2,π/2] and arccos∈[0,π] the reduced angle lies in, apply the piecewise principal-value formulas for arcsin(sin x) and arccos(cos x) (using sine/cosine symmetry and parity to choose the correct branch), then sum and simplify. | 0.7367293834686279
2: Use principal-value identity sec^{-1}x + csc^{-1}x = π/2, substitute to get a quadratic in one inverse-angle, simplify algebraically, find its vertex (complete the square or differentiate) for the minimum within the allowed principal-value interval and evaluate the interval endpoints for the maximum, then add those two values. | 1: Recall standard cosine values and their principal arccos angles, map each given value to 